# GW Glitch Utility Selection: Colab Pro continuation

**Purpose:** Continue the H1 O2 single-view baselines using the existing frozen multi-view baseline, split manifests, and processed arrays. This notebook does **not** retrain the multi-view baseline or run the O3a diagnostic.

## Before running
1. From the original Google account, share `GW_Project_Handover` and `GW_Colab_Transfer` with the Google account that has Colab Pro, with **Editor** access.
2. In the Pro account's Google Drive, add shortcuts to both shared folders in **My Drive** so they appear at:
   - `/content/drive/MyDrive/GW_Project_Handover`
   - `/content/drive/MyDrive/GW_Colab_Transfer`
3. Put `single-view-baselines-files.zip` (the ZIP Claude supplied) inside `GW_Project_Handover`.
4. Open this notebook in the Pro account and select a GPU runtime.

**Important:** The 0.5s run is already completed. The earlier 1s run was interrupted partway through. This notebook preserves its old folder in Drive, then restarts 1s cleanly rather than assuming checkpoint-resume support. It skips the completed 0.5s run and trains 1s, 2s, and 4s. Nothing uses `--overwrite` for the completed 0.5s run.

## 1. Mount Drive and confirm the Pro runtime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import torch

HANDOVER = Path('/content/drive/MyDrive/GW_Project_Handover')
TRANSFER = Path('/content/drive/MyDrive/GW_Colab_Transfer')

assert HANDOVER.is_dir(), f"Cannot find {HANDOVER}. Add a Drive shortcut to the shared handover folder."
assert TRANSFER.is_dir(), f"Cannot find {TRANSFER}. Add a Drive shortcut to the shared transfer folder."

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU assigned. In Colab: Runtime > Change runtime type > T4 GPU, then reconnect.")
print("Handover:", HANDOVER)
print("Transfer:", TRANSFER)

## 2. Clone the repository and install dependencies

In [ ]:
%cd /content
!if [ ! -d /content/GW-Glitch-Utility-Selection ]; then git clone https://github.com/amulya-714/GW-Glitch-Utility-Selection.git; fi
%cd /content/GW-Glitch-Utility-Selection

## 3. Install the single-view/evaluation additions

The ZIP must be in `GW_Project_Handover`. This cell extracts it into the repository. Re-running this cell is safe.

In [ ]:
from pathlib import Path
import zipfile

zip_path = HANDOVER / 'single-view-baselines-files.zip'
assert zip_path.exists(), (
    f"Missing {zip_path}. Upload single-view-baselines-files.zip "
    "to the shared GW_Project_Handover Drive folder first."
)

with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/GW-Glitch-Utility-Selection')

print("Single-view files extracted.")

## 4. Restore the frozen original baseline and exact processed arrays

In [ ]:
import shutil

REPO = Path('/content/GW-Glitch-Utility-Selection')
RUNS = REPO / 'experiments' / 'runs'
BASELINE = RUNS / 'H1_O2_colab_baseline_v4'
BASELINE_DRIVE = HANDOVER / 'H1_O2_baseline'

assert (BASELINE_DRIVE / 'config.yaml').exists(), "Original baseline config missing in Drive."
assert (BASELINE_DRIVE / 'checkpoints' / 'best.pt').exists(), "Original baseline best.pt missing in Drive."
shutil.copytree(BASELINE_DRIVE, BASELINE, dirs_exist_ok=True)

# Restore the large cached arrays into the repository's processed-array directory.
# The exact H1-prefixed filenames are what the patched dataset loader expects.
arr_dir = REPO / 'data' / 'processed' / 'arrays'
arr_dir.mkdir(parents=True, exist_ok=True)

for fname in ['H1_train_b3ee1fe79d42.npy', 'H1_val_680b42bea766.npy']:
    src = TRANSFER / fname
    assert src.exists(), f"Missing cached array: {src}"
    shutil.copy2(src, arr_dir / fname)
    print("Restored", fname, f"({(src.stat().st_size / 1024**3):.2f} GiB)")

print("Baseline and arrays restored.")

## 5. Download only the small O2 metadata file and create cache metadata

The cached arrays contain the expensive image preprocessing. This cell does **not** download the 73k images.

In [ ]:
import requests, json, pandas as pd, yaml

meta_dir = REPO / 'data' / 'raw' / 'metadata'
meta_dir.mkdir(parents=True, exist_ok=True)
o2_csv = meta_dir / 'H1_O2.csv'
if not o2_csv.exists():
    url = 'https://zenodo.org/records/5649212/files/H1_O2.csv?download=1'
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    o2_csv.write_bytes(r.content)
print("O2 metadata:", o2_csv, f"({o2_csv.stat().st_size/1024**2:.1f} MB)")

cfg = yaml.safe_load((REPO / 'experiments/runs/H1_O2_colab_baseline_v4/config.yaml').read_text())
split_dir = REPO / 'experiments/runs/H1_O2_colab_baseline_v4/splits'

# Recreate the .meta.json files alongside the exact cached arrays.
for split, key in [('train', 'b3ee1fe79d42'), ('val', '680b42bea766')]:
    manifest = pd.read_csv(split_dir / f'{split}_manifest.csv')
    meta = {'kept_ids': manifest['gravityspy_id'].astype(str).tolist(), 'failures': []}
    out = arr_dir / f'H1_{split}_{key}.meta.json'
    out.write_text(json.dumps(meta))
    print(out.name, "IDs:", len(meta['kept_ids']))

# Verify expected cache assets exist.
for fname in [
    'H1_train_b3ee1fe79d42.npy', 'H1_train_b3ee1fe79d42.meta.json',
    'H1_val_680b42bea766.npy', 'H1_val_680b42bea766.meta.json'
]:
    assert (arr_dir / fname).exists(), f"Missing {fname}"
print("Array cache files verified.")

## 6. Apply the two required compatibility patches

These preserve the frozen split manifests and let the data loader reuse the cached arrays **before** trying to download any images. The patch cell is idempotent.

In [ ]:
# Patch src/data/dataset.py: check for the processed array cache before downloading images.
dataset_path = REPO / 'src/data/dataset.py'
s = dataset_path.read_text()

old_dataset = '''    dl_fail = store.ensure_downloaded(df, d["view_columns"], desc=f"download[{name}]")
    if len(dl_fail):
        bad = set(dl_fail["gravityspy_id"])
        df = df.loc[~df["gravityspy_id"].isin(bad)].reset_index(drop=True)
    path, kept, dec_fail = build_split_array(
        df, store, pre, d["view_columns"], resolve_path(cfg["paths"]["processed_dir"]), name)
'''
new_dataset = '''    processed_dir = resolve_path(cfg["paths"]["processed_dir"])
    key = array_cache_key(df["gravityspy_id"].tolist(), pre, d["view_columns"])
    cache_dir = Path(processed_dir) / "arrays"
    cached_array = cache_dir / f"{name}_{key}.npy"
    cached_meta = cache_dir / f"{name}_{key}.meta.json"

    if cached_array.exists() and cached_meta.exists():
        log.info("%s: reusing processed cache; skipping image downloads", name)
        dl_fail = pd.DataFrame(columns=["gravityspy_id", "view", "url", "reason"])
    else:
        dl_fail = store.ensure_downloaded(df, d["view_columns"], desc=f"download[{name}]")
        if len(dl_fail):
            bad = set(dl_fail["gravityspy_id"])
            df = df.loc[~df["gravityspy_id"].isin(bad)].reset_index(drop=True)

    path, kept, dec_fail = build_split_array(
        df, store, pre, d["view_columns"], processed_dir, name)
'''
if old_dataset in s:
    dataset_path.write_text(s.replace(old_dataset, new_dataset, 1))
    print("Applied dataset cache-first patch.")
elif 'reusing processed cache; skipping image downloads' in s:
    print("Dataset cache-first patch already present.")
else:
    raise RuntimeError("dataset.py did not match the expected version. Stop; do not train.")

# Patch src/models/train.py: reuse the original frozen manifests/class mapping.
train_path = REPO / 'src/models/train.py'
s = train_path.read_text()
old_train = '    train_df, val_df, mapping, reports = build_source_splits(cfg)\n'
new_train = '''    reference_run = cfg.get("provenance", {}).get("reference_run")

    if reference_run:
        import json
        import pandas as pd
        from src.data.splits import ClassMapping, class_count_table

        ref_dir = resolve_path(reference_run)
        split_dir = ref_dir / "splits"
        train_df = pd.read_csv(split_dir / "train_manifest.csv")
        val_df = pd.read_csv(split_dir / "val_manifest.csv")
        mapping_data = json.loads((ref_dir / "class_mapping.json").read_text())
        mapping = ClassMapping.from_dict(mapping_data)
        purged = pd.read_csv(split_dir / "purged_events.csv")
        excluded = pd.read_csv(split_dir / "excluded_label_events.csv")
        train_df["split"] = "train"
        val_df["split"] = "val"
        reports = {
            "train_counts": class_count_table({"final(cap)": train_df}),
            "val_counts": class_count_table({"final(cap)": val_df}),
            "purged": purged,
            "outside_label_space": excluded,
        }
        log.info("Reusing frozen reference manifests: train=%d, val=%d", len(train_df), len(val_df))
    else:
        train_df, val_df, mapping, reports = build_source_splits(cfg)
'''
if old_train in s:
    assert s.count(old_train) == 1, "Unexpected number of split-building statements."
    train_path.write_text(s.replace(old_train, new_train, 1))
    print("Applied frozen-manifest patch.")
elif 'Reusing frozen reference manifests:' in s:
    print("Frozen-manifest patch already present.")
else:
    raise RuntimeError("train.py did not match the expected version. Stop; do not train.")

!python -m py_compile src/data/dataset.py src/models/train.py

## 7. Verify cache keys and the single-view CLI before training

In [ ]:
!python -m src.models.train_single_view --help

In [ ]:
# Check the exact frozen data/cache contract and current saved runs.
import numpy as np
for fname in ['H1_train_b3ee1fe79d42.npy', 'H1_val_680b42bea766.npy']:
    a = np.load(arr_dir / fname, mmap_mode='r')
    print(fname, a.shape, a.dtype)

for run_name in [
    'H1_O2_single_view_0.5s',
    'H1_O2_single_view_1s',
    'H1_O2_single_view_2s',
    'H1_O2_single_view_4s'
]:
    p = RUNS / run_name
    print(run_name, "exists:", p.exists(),
          "best:", (p/'checkpoints/best.pt').exists(),
          "last:", (p/'checkpoints/last.pt').exists())

## 8. Back up the interrupted 1s folder before restarting it

The earlier 1s run may have a partial checkpoint. Preserve it for provenance. We are not assuming the training script can resume from it.

In [ ]:
# Restore the already-completed 0.5s run if it was backed up from the previous account.
completed_05_drive = HANDOVER / 'H1_O2_single_view_0.5s'
completed_05_local = RUNS / 'H1_O2_single_view_0.5s'
if completed_05_drive.exists():
    shutil.copytree(completed_05_drive, completed_05_local, dirs_exist_ok=True)
    print("Restored completed 0.5s run from Drive.")
elif completed_05_local.exists():
    print("Completed 0.5s run already exists locally.")
else:
    print("No saved 0.5s run found. The 0.5s training cell will run it.")

# Restore any saved partial 1s run only to preserve its files for provenance.
partial_drive = HANDOVER / 'H1_O2_single_view_1s'
partial_local = RUNS / 'H1_O2_single_view_1s'
if partial_drive.exists():
    shutil.copytree(partial_drive, partial_local, dirs_exist_ok=True)

if partial_local.exists():
    archive = HANDOVER / 'H1_O2_single_view_1s_interrupted_before_restart'
    shutil.copytree(partial_local, archive, dirs_exist_ok=True)
    print("Preserved interrupted 1s run at:", archive)
else:
    print("No previous partial 1s folder found.")

## 9. Train 0.5s only if it is not already complete

This uses `--skip-existing`, so the completed 0.5s run is not overwritten.

After it finishes, run Section 13 to back up its checkpoint to Drive.

In [ ]:
!python -m src.models.train_single_view \
  --ifo H1 --view 0.5s --device cuda --workers 2 \
  --reference-run experiments/runs/H1_O2_colab_baseline_v4 \
  --skip-existing

## 10. Train 1s from scratch

The earlier 1s attempt was interrupted. This run intentionally starts fresh. Do not use this cell again after it has completed.

This restarts the interrupted 1s run cleanly. After it finishes, run Section 13 to back up its checkpoint to Drive.

In [ ]:
!python -m src.models.train_single_view \
  --ifo H1 --view 1s --device cuda --workers 2 \
  --reference-run experiments/runs/H1_O2_colab_baseline_v4 \
  --overwrite

## 11. Train 2s

**After this cell finishes, run the Drive backup cell (Section 13) before proceeding if you want an immediate checkpoint copy.**

In [ ]:
!python -m src.models.train_single_view \
  --ifo H1 --view 2s --device cuda --workers 2 \
  --reference-run experiments/runs/H1_O2_colab_baseline_v4 \
  --skip-existing

## 12. Train 4s

**After this cell finishes, run the Drive backup cell (Section 13) before proceeding if you want an immediate checkpoint copy.**

In [ ]:
!python -m src.models.train_single_view \
  --ifo H1 --view 4s --device cuda --workers 2 \
  --reference-run experiments/runs/H1_O2_colab_baseline_v4 \
  --skip-existing

## 13. Back up all results to Drive

Run this after each completed training cell if the runtime may disconnect, and once again after all four runs. It copies the run folders, including checkpoints, logs, configs, metrics, and predictions.

In [ ]:
run_names = [
    'H1_O2_colab_baseline_v4',
    'H1_O2_single_view_0.5s',
    'H1_O2_single_view_1s',
    'H1_O2_single_view_2s',
    'H1_O2_single_view_4s',
]
for name in run_names:
    src = RUNS / name
    if src.exists():
        dst = HANDOVER / name
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Backed up:", name)
    else:
        print("Not present yet:", name)

# Keep the modified source files and the run notebook's exact code version.
for rel in ['src/models/train.py', 'src/data/dataset.py']:
    dest = HANDOVER / 'colab_patches' / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO / rel, dest)
print("Drive backup complete.")

## 14. Final run inventory

In [ ]:
for name in run_names:
    p = RUNS / name
    print(
        name,
        "exists=", p.exists(),
        "best.pt=", (p/'checkpoints/best.pt').exists(),
        "last.pt=", (p/'checkpoints/last.pt').exists(),
        "Drive backup=", (HANDOVER/name).exists()
    )